# Preparing the datasets

Throughout the course, we will use the San Francisco International Airport report on monthly passenger traffic statistics by airline dataset. In this lesson we will:
- Load and review the dataset
- Prepare the dataset to work with AI agents
- Review the DuckDB approach
- Review the PostgreSQL approach 

This dataset is available at [here](https://data.sfgov.org/Transportation/Air-Traffic-Passenger-Statistics/rkru-6vcg/about_data)

## Loading the Air Passenger Traffic Dataset

Let's start by import the required libraries:

In [1]:
import pandas as pd

In [2]:
file_path = "../data/Air_Traffic_Passenger_Statistics_20260201.csv"
df = pd.read_csv(file_path)
df.head()

,Activity Period,Activity Period Start Date,Operating Airline,Operating Airline IATA Code,Published Airline,Published Airline IATA Code,GEO Summary,GEO Region,Activity Type Code,Price Category Code,Terminal,Boarding Area,Passenger Count,data_as_of,data_loaded_at
0,199907,7/1/99,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Deplaned,Low Fare,Terminal 1,B,31432,1/20/26 14:01,1/22/26 15:02
1,199907,7/1/99,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Enplaned,Low Fare,Terminal 1,B,31353,1/20/26 14:01,1/22/26 15:02
2,199907,7/1/99,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Thru / Transit,Low Fare,Terminal 1,B,2518,1/20/26 14:01,1/22/26 15:02
3,199907,7/1/99,Aeroflot Russian International Airlines,NaN,Aeroflot Russian International Airlines,NaN,International,Europe,Deplaned,Other,Terminal 2,D,1324,1/20/26 14:01,1/22/26 15:02
4,199907,7/1/99,Aeroflot Russian International Airlines,NaN,Aeroflot Russian International Airlines,NaN,International,Europe,Enplaned,Other,Terminal 2,D,1198,1/20/26 14:01,1/22/26 15:02


Let's prep the dataset to work wi with AI agents:
- Validate the column names
- Rename the column names
- Remove irrelevant columns
- Reformat the columns


In [3]:
df["Date"] = pd.to_datetime(df["Activity Period Start Date"], format="%m/%d/%y")

df["Year"] = df["Activity Period"].astype(str).str[:4].astype(int)

columns = [
    "Year",
    "Date",
    "Operating Airline",
    "Operating Airline IATA Code",
    "Published Airline",
    "Published Airline IATA Code",
    "GEO Summary",
    "GEO Region",
    "Activity Type Code",
    "Price Category Code",
    "Terminal",
    "Boarding Area",
    "Passenger Count"
]

air_traffic = df[columns].copy()
air_traffic.dtypes

Year                                    int64
Date                           datetime64[us]
Operating Airline                         str
Operating Airline IATA Code               str
Published Airline                         str
Published Airline IATA Code               str
GEO Summary                               str
GEO Region                                str
Activity Type Code                        str
Price Category Code                       str
Terminal                                  str
Boarding Area                             str
Passenger Count                         int64
dtype: object

In [4]:
air_traffic

,Year,Date,Operating Airline,Operating Airline IATA Code,Published Airline,Published Airline IATA Code,GEO Summary,GEO Region,Activity Type Code,Price Category Code,Terminal,Boarding Area,Passenger Count
0,1999,1999-07-01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Deplaned,Low Fare,Terminal 1,B,31432
1,1999,1999-07-01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Enplaned,Low Fare,Terminal 1,B,31353
2,1999,1999-07-01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Thru / Transit,Low Fare,Terminal 1,B,2518
3,1999,1999-07-01,Aeroflot Russian International Airlines,NaN,Aeroflot Russian International Airlines,NaN,International,Europe,Deplaned,Other,Terminal 2,D,1324
4,1999,1999-07-01,Aeroflot Russian International Airlines,NaN,Aeroflot Russian International Airlines,NaN,International,Europe,Enplaned,Other,Terminal 2,D,1198
...,...,...,...,...,...,...,...,...,...,...,...,...,...
39227,2025,2025-11-01,Virgin Atlantic,VS,Virgin Atlantic,VS,International,Europe,Enplaned,Other,International,A,7109
39228,2025,2025-11-01,WestJet,WS,WestJet,WS,International,Canada,Deplaned,Other,International,A,2445
39229,2025,2025-11-01,WestJet,WS,WestJet,WS,International,Canada,Enplaned,Other,International,A,2487
39230,2025,2025-11-01,ZIPAIR Tokyo Inc,ZG,ZIPAIR Tokyo Inc,ZG,International,Asia,Deplaned,Other,International,A,7279


In [5]:
air_traffic.to_csv("../data/air_traffic_gold.csv")

In [6]:
tbl_name = "air_traffic"

## DuckDB Workflow

In this section, we will review how to set up in-memory DuckDB database using the `ibis` library. Let's start by loading the `ibis` library:

In [7]:
import ibis

In [8]:
con_db = ibis.duckdb.connect()
con_db.create_table(tbl_name, air_traffic, overwrite=True)


DatabaseTable: memory.main.air_traffic
  Year                        int64
  Date                        timestamp(6)
  Operating Airline           string
  Operating Airline IATA Code string
  Published Airline           string
  Published Airline IATA Code string
  GEO Summary                 string
  GEO Region                  string
  Activity Type Code          string
  Price Category Code         string
  Terminal                    string
  Boarding Area               string
  Passenger Count             int64

In [9]:
con_db.sql("SELECT * FROM air_traffic LIMIT 10").execute()


,Year,Date,Operating Airline,Operating Airline IATA Code,Published Airline,Published Airline IATA Code,GEO Summary,GEO Region,Activity Type Code,Price Category Code,Terminal,Boarding Area,Passenger Count
0,1999,1999-07-01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Deplaned,Low Fare,Terminal 1,B,31432
1,1999,1999-07-01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Enplaned,Low Fare,Terminal 1,B,31353
2,1999,1999-07-01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Thru / Transit,Low Fare,Terminal 1,B,2518
3,1999,1999-07-01,Aeroflot Russian International Airlines,NaN,Aeroflot Russian International Airlines,NaN,International,Europe,Deplaned,Other,Terminal 2,D,1324
4,1999,1999-07-01,Aeroflot Russian International Airlines,NaN,Aeroflot Russian International Airlines,NaN,International,Europe,Enplaned,Other,Terminal 2,D,1198
5,1999,1999-07-01,Air Canada,AC,Air Canada,AC,International,Canada,Deplaned,Other,Terminal 1,B,24124
6,1999,1999-07-01,Air Canada,AC,Air Canada,AC,International,Canada,Enplaned,Other,Terminal 1,B,23613
7,1999,1999-07-01,Air China,CA,Air China,CA,International,Asia,Deplaned,Other,Terminal 2,D,4983
8,1999,1999-07-01,Air China,CA,Air China,CA,International,Asia,Enplaned,Other,Terminal 2,D,4604
9,1999,1999-07-01,Air Europe,PE,Air Europe,PE,International,Europe,Deplaned,Other,Terminal 2,D,205


## Postgres Workflow

In [10]:
con_postgres = ibis.postgres.connect(
    user="postgres",
    password="password",
    host="localhost",
    port=5432,
    database="my_db",
)

In [11]:
schema = ibis.memtable(air_traffic).schema()
print(schema)

ibis.Schema {
  Year                         int64
  Date                         timestamp
  Operating Airline            string
  Operating Airline IATA Code  string
  Published Airline            string
  Published Airline IATA Code  string
  GEO Summary                  string
  GEO Region                   string
  Activity Type Code           string
  Price Category Code          string
  Terminal                     string
  Boarding Area                string
  Passenger Count              int64
}


In [12]:
con_postgres.create_table("air_traffic", air_traffic, schema=schema, overwrite=True)

DatabaseTable: air_traffic
  Year                        int64
  Date                        timestamp
  Operating Airline           string
  Operating Airline IATA Code string
  Published Airline           string
  Published Airline IATA Code string
  GEO Summary                 string
  GEO Region                  string
  Activity Type Code          string
  Price Category Code         string
  Terminal                    string
  Boarding Area               string
  Passenger Count             int64

In [13]:
con_postgres.sql("SELECT * FROM air_traffic LIMIT 10").execute()


,Year,Date,Operating Airline,Operating Airline IATA Code,Published Airline,Published Airline IATA Code,GEO Summary,GEO Region,Activity Type Code,Price Category Code,Terminal,Boarding Area,Passenger Count
0,1999,1999-07-01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Deplaned,Low Fare,Terminal 1,B,31432
1,1999,1999-07-01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Enplaned,Low Fare,Terminal 1,B,31353
2,1999,1999-07-01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Thru / Transit,Low Fare,Terminal 1,B,2518
3,1999,1999-07-01,Aeroflot Russian International Airlines,NaN,Aeroflot Russian International Airlines,NaN,International,Europe,Deplaned,Other,Terminal 2,D,1324
4,1999,1999-07-01,Aeroflot Russian International Airlines,NaN,Aeroflot Russian International Airlines,NaN,International,Europe,Enplaned,Other,Terminal 2,D,1198
5,1999,1999-07-01,Air Canada,AC,Air Canada,AC,International,Canada,Deplaned,Other,Terminal 1,B,24124
6,1999,1999-07-01,Air Canada,AC,Air Canada,AC,International,Canada,Enplaned,Other,Terminal 1,B,23613
7,1999,1999-07-01,Air China,CA,Air China,CA,International,Asia,Deplaned,Other,Terminal 2,D,4983
8,1999,1999-07-01,Air China,CA,Air China,CA,International,Asia,Enplaned,Other,Terminal 2,D,4604
9,1999,1999-07-01,Air Europe,PE,Air Europe,PE,International,Europe,Deplaned,Other,Terminal 2,D,205
